# Python Core Mechanics

Five foundational Python patterns — applied to trading and finance contexts.
Each section includes explanation, annotated code, and a mini challenge.

| # | Mechanic | Why it matters |
|---|----------|----------------|
| 1 | Loops + `enumerate` | Avoid manual index bugs, iterate with position |
| 2 | Dictionaries | O(1) lookup — core of frequency maps, grouping, caching |
| 3 | Sorting | `key=` unlocks sorting anything by any field |
| 4 | List comprehensions | Replace map/filter with readable one-liners |
| 5 | Strings | Parse, format, and split structured text data |
| 6 | Slicing | Extract subsequences from strings, lists, and any sequence |

---

## Mechanic 1: Loops + `enumerate`

`enumerate(iterable)` yields `(index, value)` pairs — you never need a manual counter variable.

```python
# Without enumerate (fragile)
i = 0
for item in items:
    print(i, item)
    i += 1

# With enumerate (clean)
for i, item in enumerate(items):
    print(i, item)
```

**Syntax:** `for i, item in enumerate(iterable, start=0):`

The optional `start=` argument offsets the counter — useful for 1-indexed output or when the index should correspond to a domain concept (e.g., "Candle 1" not "Candle 0").

**Interview angle:** Reach for `enumerate` any time you need both the element *and* its position. It eliminates off-by-one bugs and is the idiomatic alternative to `range(len(lst))`. Common in sliding window problems, finding the index of a max/min, or building a value-to-index map.

**Trading context:** Labeling bars by position, detecting bar-over-bar changes (`prices[i]` vs `prices[i-1]`), or building a time-indexed event list.

In [ ]:
prices = [101.5, 98.2, 103.7, 99.1, 105.0]

# Basic enumerate
print("Bar data (0-indexed):")
for i, price in enumerate(prices):
    print(f"  Bar {i}: ${price}")

print()

# Offset start for 1-indexed candle labels
print("Candle labels (1-indexed):")
for i, price in enumerate(prices, start=1):
    print(f"  Candle {i}: ${price}")

In [ ]:
# Mini challenge: flag bars where price crossed above $100
print("Bars above $100 (with position):")
for i, price in enumerate(prices):
    if price > 100:
        status = '\u25b2 ABOVE' if price > 100 else '\u25bc BELOW'
        print(f"  Bar {i}: ${price}  {status}")

# Practical: detect where price drops below previous bar
print("\nDownbars:")
for i, price in enumerate(prices[1:], start=1):
    if price < prices[i - 1]:
        print(f"  Bar {i}: ${price} < Bar {i-1}: ${prices[i-1]}  \u2193")

## Mechanic 2: Dictionaries

Dictionaries are Python's **hash map** — O(1) average insert and lookup.

Key patterns:
- `.get(key, default)` — safe access without KeyError
- `.items()` — iterate key-value pairs
- Counting pattern: `d[k] = d.get(k, 0) + 1`
- Grouping pattern: `d.setdefault(k, []).append(v)`

### `dict.get()` and `.items()`

**`dict.get(key, default)`** returns the value for `key` if it exists, otherwise `default` — never raises `KeyError`. Omitting the default falls back to `None`.

**Syntax:** `value = d.get('key', fallback_value)`

**`dict.items()`** returns `(key, value)` pairs as an iterable. Write `for k, v in d.items():` instead of `for k in d: v = d[k]` — it's faster and signals intent.

**When to use `get()`:** Any time a key might be absent — missing tickers, config defaults, P&L of 0 when no trade recorded. It's also the building block for the counting pattern in the next cell.

**Interview angle:** `dict.get()` is the idiomatic way to handle optional keys without a try/except or an `if key in d` guard. If you're writing those repeatedly, `get()` is the cleaner path.

```python
# The problem with direct access:
pnl = {'NQ': 1500}
pnl['ES']         # KeyError — crashes if key absent

# The solution:
pnl.get('ES', 0)  # → 0, safe
```

In [ ]:
# dict.get() — safe key access
pnl = {'NQ': 1500, 'ES': -200}
tickers = ['NQ', 'ES', 'GC']   # GC has no P&L recorded

for ticker in tickers:
    # get() returns 0 for GC instead of raising KeyError
    print(f"  {ticker} P&L: ${pnl.get(ticker, 0):,}")

# .items() — iterate key-value pairs cleanly
print()
portfolio = {'NQ': 2, 'ES': 1, 'GC': 3}
for ticker, qty in portfolio.items():
    print(f"  {ticker}: {qty} contracts")

In [ ]:
portfolio = {
    "NQ": {"qty": 2, "entry": 19800},
    "ES": {"qty": 1, "entry": 5050},
    "GC": {"qty": 3, "entry": 2350},
}

# Iterate with .items()
print("Current portfolio:")
for ticker, info in portfolio.items():
    print(f"  {ticker}: {info['qty']} contracts @ {info['entry']:,}")

# Safe access — no KeyError if key is missing
pnl = {"NQ": 1500, "ES": -200}
print()
for ticker in portfolio:
    print(f"  {ticker} P&L: ${pnl.get(ticker, 0):,}")

### Counting pattern and `dict.setdefault()`

**Counting/frequency map** — `d[k] = d.get(k, 0) + 1` increments a counter for `k`, initializing it to 0 on first encounter. This is the standard frequency map without importing `Counter`.

```python
d = {}
d['BUY'] = d.get('BUY', 0) + 1  # d = {'BUY': 1}
d['BUY'] = d.get('BUY', 0) + 1  # d = {'BUY': 2}
```

**`dict.setdefault(key, default)`** — if `key` exists, return its value unchanged. If absent, insert `key: default` *and* return `default`. The key insight: it both sets and returns in one call, so you can chain `.append()` directly.

```python
groups = {}
groups.setdefault('NQ', []).append(500)   # {'NQ': [500]}
groups.setdefault('NQ', []).append(800)   # {'NQ': [500, 800]}
groups.setdefault('ES', []).append(-200)  # {'NQ': [500, 800], 'ES': [-200]}
```

**Interview angle:** These two patterns cover a massive slice of hash map problems. Any "group by" operation → `setdefault`. Any "count occurrences" → `d.get(k, 0) + 1`. Internalize both and you'll handle most "use a hash map" hints without hesitation.

In [ ]:
# Counting pattern (frequency map)
trades = ['BUY', 'SELL', 'BUY', 'BUY', 'SELL']
freq = {}
for t in trades:
    freq[t] = freq.get(t, 0) + 1
print(f"Frequency map: {freq}")   # {'BUY': 3, 'SELL': 2}

# Grouping pattern with setdefault
trade_log = [('NQ', 500), ('ES', -200), ('NQ', 800), ('ES', 300)]
by_ticker = {}
for ticker, pnl in trade_log:
    by_ticker.setdefault(ticker, []).append(pnl)
print(f"Grouped:       {by_ticker}")

# Derived metrics from the groups
print()
for ticker, pnls in by_ticker.items():
    print(f"  {ticker}: total={sum(pnls):+,}  count={len(pnls)}")

In [ ]:
# Counting pattern — frequency map
trades = ["BUY", "SELL", "BUY", "BUY", "SELL", "BUY", "SELL", "SELL", "SELL"]
count = {}
for t in trades:
    count[t] = count.get(t, 0) + 1
print(f"Trade counts: {count}")
win_rate = count.get('BUY', 0) / len(trades)
print(f"Buy rate: {win_rate:.1%}")

# Grouping pattern — group trades by direction
trade_log = [
    {"ticker": "NQ", "dir": "BUY",  "pnl": 500},
    {"ticker": "ES", "dir": "SELL", "pnl": -200},
    {"ticker": "NQ", "dir": "BUY",  "pnl": 800},
    {"ticker": "GC", "dir": "BUY",  "pnl": -100},
    {"ticker": "ES", "dir": "SELL", "pnl": 300},
]
by_ticker = {}
for trade in trade_log:
    by_ticker.setdefault(trade['ticker'], []).append(trade['pnl'])

print("\nP&L by ticker:")
for ticker, pnls in by_ticker.items():
    print(f"  {ticker}: total=${sum(pnls):,}  trades={len(pnls)}")

## Mechanic 3: Sorting

- `sorted(iterable)` — returns a **new** sorted list, original unchanged
- `list.sort()` — sorts **in place**, returns `None`
- `key=` accepts any callable — use `lambda` for inline extraction logic
- `reverse=True` for descending order

**Syntax:**
```python
sorted(items, key=lambda x: x['field'], reverse=True)
```

**How `key=` works:** Python calls `key(element)` on each item to produce a comparison value, sorts by those values, then reconstructs the list. The original objects are untouched — you're sorting *by* the extracted value, not replacing it.

**Interview angle:** `key=` is Python's general-purpose sort hook. Once you internalize "pass a function that extracts what you want to compare," you can sort dicts, objects, or tuples by any field. This shows up constantly in problems involving rankings, leaderboards, and scheduling by priority.

**Trading context:** Ranking signals by conviction score, top-N positions by notional exposure, trade history sorted by P&L or timestamp.

In [ ]:
# Basic sorts
closes = [103.2, 98.5, 101.0, 99.8, 105.3, 97.1]
print(f"Original:   {closes}")
print(f"Ascending:  {sorted(closes)}")
print(f"Descending: {sorted(closes, reverse=True)}")

# Sort list of dicts by field
signals = [
    {"ticker": "NQ", "score": 0.82, "direction": "LONG"},
    {"ticker": "ES", "score": 0.65, "direction": "SHORT"},
    {"ticker": "GC", "score": 0.91, "direction": "LONG"},
    {"ticker": "ZB", "score": 0.44, "direction": "SHORT"},
]
ranked = sorted(signals, key=lambda s: s['score'], reverse=True)
print("\nSignals ranked by score:")
for rank, s in enumerate(ranked, 1):
    print(f"  #{rank} {s['ticker']} ({s['direction']}) \u2014 score {s['score']}")

### Multi-key sorting and sorting dict items

**Multi-key sort** — return a **tuple** from `key=`. Python compares tuples element-by-element: first field first, second field as the tiebreaker.

```python
sorted(items, key=lambda x: (x['primary'], x['secondary']))
```

**Descending on a secondary key — the negation trick:** There's no per-field `reverse=` parameter. Negate numeric fields with `-` to flip their individual ordering:

```python
# Group by direction (A→Z), then score highest-first within each group
sorted(signals, key=lambda s: (s['direction'], -s['score']))
```

**`sorted(dict.items(), ...)`** — dicts aren't directly sortable by value. `.items()` gives you `(key, value)` tuples; index into the tuple inside the lambda:

```python
sorted(d.items(), key=lambda x: x[1]['qty'] * x[1]['entry'], reverse=True)
# x[0] = dict key (ticker), x[1] = dict value (inner dict)
```

In [ ]:
# Multi-key sort: primary by direction, secondary by score descending
multi_sorted = sorted(signals, key=lambda s: (s['direction'], -s['score']))
print("Sorted by direction then score desc:")
for s in multi_sorted:
    print(f"  {s['direction']:<6} {s['ticker']}  score={s['score']}")

# Mini challenge: sort portfolio by total notional value (qty * entry)
portfolio = {
    "NQ": {"qty": 2, "entry": 19800},
    "ES": {"qty": 1, "entry": 5050},
    "GC": {"qty": 3, "entry": 2350},
}
by_notional = sorted(
    portfolio.items(),
    key=lambda x: x[1]['qty'] * x[1]['entry'],
    reverse=True
)
print("\nPortfolio by notional value:")
for ticker, info in by_notional:
    notional = info['qty'] * info['entry']
    print(f"  {ticker}: {info['qty']} x {info['entry']:,} = ${notional:,}")

## Mechanic 4: List Comprehensions

Compact syntax for building lists — replaces most `for` loops that append to a list.

```python
[expression for item in iterable if condition]
```

- The `if condition` is optional (filter step)
- Dict comprehensions: `{k: v for k, v in ...}`
- Set comprehensions: `{x for x in ...}`
- Nested comprehensions: two `for` clauses in one expression

**Interview angle:** Fluent comprehension syntax signals Python fluency. Use them for transforms and filters on existing sequences. The dict comprehension `{k: f(v) for k, v in d.items()}` is particularly common for remapping dict values.

**Trading context:** Computing returns from a price series, filtering signals by multiple criteria, building ticker→metric lookup dicts, generating all pairwise combinations for a correlation matrix.

In [ ]:
# Basic: squares
squares = [x**2 for x in range(1, 11)]
print(f"Squares: {squares}")

# Filtered
even_squares = [x**2 for x in range(1, 11) if x % 2 == 0]
print(f"Even squares: {even_squares}")

# Applied to price data: compute bar-over-bar returns
prices = [101.5, 98.2, 103.7, 99.1, 105.0]
returns = [(prices[i] - prices[i-1]) / prices[i-1] for i in range(1, len(prices))]
print(f"\nReturns: {[round(r, 4) for r in returns]}")
print(f"Avg return: {sum(returns)/len(returns):.4f}")
print(f"Down bars:  {sum(1 for r in returns if r < 0)} of {len(returns)}")

### Dict comprehensions and nested comprehensions

**Dict comprehension** — same concept as list comprehension, produces a dict instead of a list:

```python
{key_expr: value_expr for item in iterable if condition}
```

```python
portfolio = {'NQ': {'qty': 2, 'entry': 19800}, 'ES': {'qty': 1, 'entry': 5050}}
notional = {t: info['qty'] * info['entry'] for t, info in portfolio.items()}
# → {'NQ': 39600, 'ES': 5050}
```

**Nested comprehension** — two `for` clauses in one expression. The outer loop runs first; the inner loop runs for each outer iteration. Use for flattening 2D data or generating combinations:

```python
tickers = ['NQ', 'ES', 'GC']
# All unique pairs — enumerate gives index so inner loop starts after current item
pairs = [(a, b) for i, a in enumerate(tickers) for b in tickers[i+1:]]
# → [('NQ', 'ES'), ('NQ', 'GC'), ('ES', 'GC')]
```

**When to use:** Dict comprehensions shine for remapping/transforming dicts (apply a function to all values, swap keys and values, filter to a subset). Nested comprehensions are best limited to two levels — deeper gets unreadable.

In [ ]:
# Dict comprehension: ticker -> notional value
portfolio = {
    "NQ": {"qty": 2, "entry": 19800},
    "ES": {"qty": 1, "entry": 5050},
    "GC": {"qty": 3, "entry": 2350},
}
notional = {t: info['qty'] * info['entry'] for t, info in portfolio.items()}
print(f"Notional values: {notional}")

# Nested comprehension: all (i,j) trade pairs for correlation analysis
tickers = ['NQ', 'ES', 'GC']
pairs = [(a, b) for i, a in enumerate(tickers) for b in tickers[i+1:]]
print(f"\nTicker pairs for correlation: {pairs}")

# Mini challenge: filter signals with score > 0.7 and direction LONG
signals = [
    {"ticker": "NQ", "score": 0.82, "direction": "LONG"},
    {"ticker": "ES", "score": 0.65, "direction": "SHORT"},
    {"ticker": "GC", "score": 0.91, "direction": "LONG"},
    {"ticker": "ZB", "score": 0.44, "direction": "SHORT"},
]
qualified = [s['ticker'] for s in signals if s['score'] > 0.70 and s['direction'] == 'LONG']
print(f"\nQualified LONG signals (score > 0.70): {qualified}")

## Mechanic 5: Strings

Strings are **immutable sequences** — every method returns a *new* string, the original is unchanged.

Key methods:

| Method | What it does |
|--------|-------------|
| `.strip()` | Remove leading/trailing whitespace |
| `.split(sep)` | Split into list on separator |
| `sep.join(list)` | Join list back into string |
| `.upper()` / `.lower()` | Case conversion |
| `.startswith()` / `.endswith()` | Prefix/suffix check |
| `.replace(old, new)` | Substitute substring |
| `f"{value:.2f}"` | Formatted string literals |

**Interview angle:** String parsing comes up constantly — splitting CSV rows, normalizing tickers, extracting fields from log lines. `split()` + `join()` is the most common pairing. f-strings with format specs are essential for clean tabular output.

### `str.split()` and `str.join()`

**`str.split(sep)`** splits a string on `sep` and returns a list. Called on the separator, not the string.

**Syntax:** `parts = "a,b,c".split(",")` → `['a', 'b', 'c']`

**`sep.join(list)`** is the inverse — join a list of strings with `sep` as the glue. Called on the separator, not the list.

**Syntax:** `",".join(['a', 'b', 'c'])` → `"a,b,c"`

**Optional `maxsplit` argument:** `s.split(sep, maxsplit)` splits at most `maxsplit` times — useful when you only want to split on the first occurrence:

```python
"LONG_NQ_20240115".split("_", 1)  # ['LONG', 'NQ_20240115'] — splits once only
```

**Interview angle:** `split()` turns a raw string into a list you can index into and manipulate. `join()` reassembles it. These two together handle most string-to-structured-data parsing tasks.

In [ ]:
# split() — string to list
csv_row = "NQ,2,19800,LONG"
parts = csv_row.split(",")
print(parts)            # ['NQ', '2', '19800', 'LONG']
print(parts[0])         # 'NQ'
print(parts[2])         # '19800'

# join() — list to string
print(" | ".join(parts))   # 'NQ | 2 | 19800 | LONG'
print("-".join(parts))     # 'NQ-2-19800-LONG'

# maxsplit: split only on first separator
tag = "LONG_NQ_20240115"
direction, rest = tag.split("_", 1)
print(f"direction={direction}, rest={rest}")   # direction=LONG, rest=NQ_20240115

In [ ]:
# Clean and normalize raw input
raw = "  NQ Futures  "
clean = raw.strip()
print(f"Stripped: '{clean}'")
print(f"Upper:    '{clean.upper()}'")
print(f"Lower:    '{clean.lower()}'")

# Parse CSV trade data
csv_row = "NQ,2,19800,LONG,2024-01-15"
ticker, qty, entry, direction, date = csv_row.split(",")
print(f"\nParsed: ticker={ticker} qty={qty} entry={entry} dir={direction} date={date}")

# Rejoin with different separator
print(f"Formatted: {' | '.join([ticker, qty, entry, direction])}") 

### f-string format specifiers

The format spec goes after `:` inside `{}` — it controls width, alignment, number format, and precision.

**Syntax:** `f"{value : [fill][align][width][,][.precision][type]}"`

| Spec | Meaning | Example output |
|------|---------|---------------|
| `>12` | right-align in 12-char field | `'          NQ'` |
| `,` | thousands separator | `19,800` |
| `.2f` | float, 2 decimal places | `3.14` |
| `>12,.2f` | combined: right-aligned, comma, 2 decimals | `  19,842.50` |
| `.1%` | multiply by 100, show as %, 1 decimal | `62.3%` |
| `+` | always show sign | `+500`, `-200` |

**When to use:** Any time you need aligned tabular output or clean monetary/percentage display. The `,.2f` combo is standard for prices and P&L. The `>N` width spec is what makes columns line up.

```python
price = 19842.50
print(f"Entry: {price:>12,.2f}")   # 'Entry:     19,842.50'
print(f"Win %: {0.623:.1%}")       # 'Win %: 62.3%'
```

In [ ]:
# f-string format spec examples — run these to see the output
price = 19842.50
pct   = 0.6231
pnl   = 1500

print(f"Basic:          {price}")           # no formatting
print(f"2 decimals:     {price:.2f}")       # 19842.50
print(f"Thousands:      {price:,.2f}")      # 19,842.50
print(f"Right-aligned:  {price:>15,.2f}")   # right-pad to 15 chars
print(f"Percentage:     {pct:.1%}")         # 62.3%
print(f"Signed P&L:     {pnl:+,}")         # +1,500

# Tabular output — the '>N' width makes columns align
print()
trades = [("NQ", 1500), ("ES", -200), ("GC", 875)]
print(f"  {'Ticker':<8} {'P&L':>10}")
print(f"  {'-'*8} {'-'*10}")
for ticker, p in trades:
    print(f"  {ticker:<8} {p:>+10,}")

In [ ]:
# f-strings with format specs
entry = 19842.50
stop  = 19750.00
risk  = entry - stop
size  = 2
dollar_risk = risk * size * 20  # NQ point value = $20

print(f"Entry:        {entry:>12,.2f}")
print(f"Stop:         {stop:>12,.2f}")
print(f"Risk (pts):   {risk:>12.2f}")
print(f"Dollar risk:  {dollar_risk:>12,.2f}")

# Mini challenge: parse 'TICKER_DIRECTION' tags into structured dict
tags = ["LONG_NQ", "SHORT_ES", "LONG_GC", "FLAT_ZB"]
parsed = {}
for tag in tags:
    direction, ticker = tag.split("_", 1)
    parsed[ticker] = direction
print(f"\nParsed tags: {parsed}")

# Filter to only active (non-FLAT) positions
active = {t: d for t, d in parsed.items() if d != 'FLAT'}
print(f"Active positions: {active}")

## Mechanic 6: Slicing

Slicing extracts a subsequence using `[start:stop:step]` — works on strings, lists, tuples, and any sequence. All three parts are optional.

**Syntax:** `sequence[start:stop:step]`

| Part | Default | Meaning |
|------|---------|---------|
| `start` | `0` | Inclusive index where slice begins |
| `stop` | `len(s)` | Exclusive index where slice ends |
| `step` | `1` | Stride between elements; negative reverses direction |

**Key patterns:**

```python
s = "NQ_LONG_20240115"
s[:2]       # 'NQ'    — first 2 chars (stop only)
s[-8:]      # '20240115' — last 8 chars (negative start)
s[3:7]      # 'LONG'  — chars at index 3, 4, 5, 6
s[::-1]     # reverse the entire string
```

**Negative indices:** Count from the end. `s[-1]` is the last element, `s[-3:]` is the last three. This is the idiomatic way to take "tail" windows.

**Interview angle:** Slicing drives nearly every string-parsing problem and most array window problems. `arr[-n:]` for the last N elements, `arr[::2]` for every other element, and `arr[::-1]` for reversal are the three patterns you'll use most.

**Trading context:** Parsing fixed-format date strings (`date_str[:4]` for year), tail windows for recent price data (`prices[-20:]` for last 20 bars), reversing arrays for lookback scans.

In [ ]:
# String slicing — fixed-format parsing
trade_str = "LONG_NQ_20240115"
direction = trade_str[:4]       # 'LONG'
ticker    = trade_str[5:7]      # 'NQ'
date      = trade_str[-8:]      # '20240115'
year      = trade_str[-8:-4]    # '2024'
print(f"direction={direction}  ticker={ticker}  date={date}  year={year}")

# List slicing — price data windows
prices = [101.5, 98.2, 103.7, 99.1, 105.0, 107.2, 104.8]

last_3       = prices[-3:]      # last 3 bars
first_3      = prices[:3]       # first 3 bars
middle       = prices[2:5]      # index 2, 3, 4
every_other  = prices[::2]      # bars 0, 2, 4, 6
reversed_p   = prices[::-1]     # newest first

print(f"\nAll bars:    {prices}")
print(f"Last 3:      {last_3}")
print(f"First 3:     {first_3}")
print(f"Middle [2:5]:{middle}")
print(f"Every other: {every_other}")
print(f"Reversed:    {reversed_p}")

# Mini challenge: is the last 3-bar average higher than the first 3-bar average?
avg_tail = sum(prices[-3:]) / 3
avg_head = sum(prices[:3]) / 3
trend = "\u25b2 upward" if avg_tail > avg_head else "\u25bc downward"
print(f"\nHead avg={avg_head:.2f}  Tail avg={avg_tail:.2f}  Trend: {trend}")

---
## Quick Reference

```python
# enumerate
for i, item in enumerate(lst, start=1): ...

# dict safe access + counting
d.get(key, default)
d[k] = d.get(k, 0) + 1
d.setdefault(k, []).append(v)

# sorting by field
sorted(items, key=lambda x: x['field'], reverse=True)
# multi-key: primary asc, secondary desc
sorted(items, key=lambda x: (x['primary'], -x['secondary']))

# comprehension with filter
[f(x) for x in lst if condition(x)]
{k: v for k, v in d.items() if condition}

# string split/join
parts = s.split(',')         # str -> list
parts = s.split('_', 1)      # split on first '_' only
s     = ','.join(parts)      # list -> str

# slicing
s[-n:]    # last n elements
s[:n]     # first n elements
s[::-1]   # reversed
s[i:j]    # elements i up to (not including) j
```